In [ ]:
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

# 1. Montage de Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monté avec succès.')
except ImportError:
    print('Environnement local détecté. Assurez-vous que les chemins sont corrects.')

In [ ]:
# 2. Configuration des chemins
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021')

CSV_FILES = [
    DRIVE_BASE_DIR / 'mctnet_samples_AR_2021.csv',
    DRIVE_BASE_DIR / 'mctnet_samples_CA_2021.csv',
]

OUTPUT_DIR = DRIVE_BASE_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Le script build_dataset.py doit être dans le même dossier ou accessible
!python /content/drive/MyDrive/mctnet_crop_mapping_2021/build_dataset.py \
    --input-csv {CSV_FILES[0]} {CSV_FILES[1]} \
    --output-dir {OUTPUT_DIR}

## Validation des Tenseurs pour PyTorch
Vérification que les dimensions sont conformes aux attentes de l'architecture CNN-Transformer (N échantillons, 36 pas de temps, 10 bandes).

In [ ]:
class CropTimeSeriesDataset(Dataset):
    def __init__(self, bundle, split):
        self.x = torch.from_numpy(bundle[f'x_{split}']).float()
        self.valid_mask = torch.from_numpy(bundle[f'valid_mask_{split}']).float()
        self.missing_mask = torch.from_numpy(bundle[f'missing_mask_{split}']).float()
        self.y = torch.from_numpy(bundle[f'y_{split}']).long()

    def __len__(self):
        return int(self.x.shape[0])

    def __getitem__(self, index):
        return {
            'x': self.x[index],
            'valid_mask': self.valid_mask[index],
            'missing_mask': self.missing_mask[index],
            'y': self.y[index],
        }

def validate_dataloader(npz_path, split, batch_size=32):
    with np.load(npz_path, allow_pickle=True) as data:
        bundle = {key: data[key] for key in data.files}
        
    dataset = CropTimeSeriesDataset(bundle, split)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    batch = next(iter(loader))

    # Vérification stricte des dimensions
    assert batch['x'].shape[1:] == (36, 10), f"Erreur X: {batch['x'].shape}"
    assert batch['valid_mask'].shape[1:] == (36,), f"Erreur Valid Mask: {batch['valid_mask'].shape}"
    assert batch['missing_mask'].shape[1:] == (36, 10), f"Erreur Missing Mask: {batch['missing_mask'].shape}"
    
    print(f'[{split.upper()}] Validation Réussie | X: {tuple(batch["x"].shape)} | Y: {tuple(batch["y"].shape)}')


In [ ]:
dataset_files = sorted(OUTPUT_DIR.glob('*_mctnet_dataset.npz'))

for npz_path in dataset_files:
    state_slug = npz_path.stem.replace('_mctnet_dataset', '')
    json_path = OUTPUT_DIR / f'{state_slug}_mctnet_dataset.json'
    metadata = json.loads(json_path.read_text(encoding='utf-8'))

    print('=' * 60)
    print(f"Validation pour : {metadata['state_name'].upper()}")
    print(f"Classes : {metadata['class_name_to_index']}")
    print('-' * 60)
    
    for split in ['train', 'val', 'test']:
        validate_dataloader(npz_path, split)
